# Testing MPNN Models trained with spice and SPICE Datasets

## spice Dataset

In [ ]:
from torch.utils.data import random_split
import numpy as np
import importlib
import logging

logging.basicConfig(
    level=logging.INFO,
    format='[%(asctime)s] %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("dataset_loader.log"),
        logging.StreamHandler()
    ]
)

In [ ]:
from atoMLtype.datasets.GNNdataset import GNNdataset
from atoMLtype.models.ModelEncoder import ModelEncoder

spice_dataset_encoder = ModelEncoder(collapse=True)

# Load the spice files
spice_sdf_path = "./data/SPICE/SPICE_batch_0_2.sdf"
spice_json_labels = "./data/SPICE/SPICE_batch_0_2_gaff2.json"

# Initialize dataset
spice_dataset = GNNdataset(spice_sdf_path, 
                              spice_json_labels, 
                              directed_graph=True, 
                              labeled=True,
                              encoder=spice_dataset_encoder)

In [ ]:
from atoMLtype.analysis.accuracy_counts import plot_atom_distribution

# Split Train and test dataset (90% train, 10% test)
train_size = int(0.90 * len(spice_dataset))
test_size = len(spice_dataset) - train_size
train_dataset, test_dataset = random_split(spice_dataset, [train_size, test_size])

print("FULL DATASET (encoded):")
plot_atom_distribution(np.array(spice_dataset_encoder.inverse_transform(spice_dataset.encoded_labels)))

In [ ]:
from atoMLtype.models.GNN.DMPNNmodel import AtomBondMPNN
from atoMLtype.models.ModelTrainer import GNNTrainer

AtomMPNN_spice = AtomBondMPNN(atom_input_dim=train_dataset[0].x.shape[1], 
                                      bond_input_dim=train_dataset[0].edge_attr.shape[1], 
                                      hidden_dim=512,
                                      encoder=spice_dataset_encoder, 
                                      num_layers=10,
                                      use_attention=True)

trainer_AtomMPNN_spice = GNNTrainer(AtomMPNN_spice, 
                                   dataset=train_dataset, 
                                   batch_size=32, learning_rate=0.001,
                                   epochs=15, 
                                   k_folds=5, 
                                   random_seed=21)

loss_ouput = trainer_AtomMPNN_spice.train(verbose=True)



In [ ]:
from atoMLtype.models.ModelEngine import ModelEngine

modelEngine_spice = ModelEngine(model=AtomMPNN_spice, 
                          dataset=test_dataset, 
                          device="cpu",
                          batch_size=32)

spice_record = modelEngine_spice.predict(analysis=True)


In [ ]:
spice_record.summary()

In [ ]:
AtomMPNN_spice.save("./saved_models/AtomMPNN_Spice.mdl")

In [ ]:
from atoMLtype.analysis.confusionMatrices import plot_full_confusion_matrices

plot_full_confusion_matrices(spice_record)

In [ ]:
from atoMLtype.analysis.discrepancies import plot_confidence_by_pred_label, plot_discrepancy_distribution, plot_element_discrepancy_rate, plot_discrepancy_rate_by_atom_type


plot_confidence_by_pred_label(spice_record, 
                              sort_by='alphabetical',
                              show_mismatch=True,
                              showfliers=True, 
                              figsize=(10, 5))

plot_discrepancy_distribution(spice_record)

valid_elements = {
        "f", "cl", "br", "i", "c", "h", "n", "o", "s", "p"
    }

plot_element_discrepancy_rate(spice_record, valid_elements)

plot_discrepancy_rate_by_atom_type(spice_record)

In [ ]:
from atoMLtype.analysis.molecule_embeddings import visualize_prediction_embeddings

visualize_prediction_embeddings(
    pred_record=spice_record,
    key='clf_embeddings',
    method='tsne'
)

In [ ]:
len(spice_record.mismatched_molecules)

In [ ]:
from atoMLtype.analysis.molecule_embeddings import draw_molecule_with_mismatches_labeled


for mol_name, atom_preds in spice_record.mismatched_molecules.items():
    for atom_pred in atom_preds:
        if not atom_pred.pred_label == atom_pred.true_label:
            print(f"{mol_name} atom {atom_pred.atom_idx_in_mol}, pred AT {atom_pred.pred_label} for {atom_pred.true_label}")
    mol = spice_dataset.get_mol(mol_name)

    img = draw_molecule_with_mismatches_labeled(mol, atom_preds)
    display(img)  # or img.save(f"{mol_name}_mismatch.png")